#**Youtube Video Summery Generation Tool**

In [ ]:
# Log in to the Hugging Face Hub
from huggingface_hub import login

login("hugging_face_api")

In [ ]:
# Load environment variables and Hugging Face token
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("hugging_face_api")

In [ ]:
# Install necessary Python packages
!pip install -q \
youtube-transcript-api \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
llama-cpp-python \
huggingface_hub \
sentence-transformers \
faiss-cpu \
tiktoken \
python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# Import required libraries for RAG pipeline
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from huggingface_hub import hf_hub_download

/tmp/ipykernel_508/1466627329.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


In [ ]:
# Download a list of free proxies
import requests

url = (
    "https://api.proxyscrape.com/v4/free-proxy-list/get"
    "?request=display_proxies"
    "&protocol=http"
    "&proxy_format=ipport"
    "&format=text"
)

proxy_list = requests.get(url).text.splitlines()

print(f"Downloaded {len(proxy_list)} proxies")
print(proxy_list[:10])

Downloaded 807 proxies
['104.194.148.204:80', '91.188.213.143:1080', '85.234.100.149:8080', '185.161.251.195:3128', '94.158.49.82:3128', '71.198.208.169:443', '103.167.61.162:3128', '219.93.101.62:80', '47.91.65.23:3128', '120.92.212.16:8890']


In [ ]:
# Function to find and select a working proxy
import requests

def get_working_proxy(proxy_list):
    test_url = "http://httpbin.org/ip"

    for proxy in proxy_list:
        proxy_url = f"http://{proxy}"

        try:
            r = requests.get(
                test_url,
                proxies=
                    {
                    "http": proxy_url,
                    "https": proxy_url
                },
                timeout=5
            )

            if r.status_code == 200:
                print("✅ Working:", proxy)
                return proxy_url

        except:
            continue

    return None

working_proxy = get_working_proxy(proxy_list)
print("Selected:", working_proxy)

✅ Working: 43.133.1.198:3128
Selected: http://43.133.1.198:3128


In [ ]:
# Fetch YouTube transcript using the working proxy
from requests import Session
from youtube_transcript_api import YouTubeTranscriptApi

proxy = "http://195.158.8.123:3128"

session = Session()
session.proxies = {
    "http": proxy,
    "https": proxy
}

api = YouTubeTranscriptApi(http_client=session)

transcript = api.fetch(
    "Gfr50f6ZBvo",
    languages=["en"]
)

# Convert transcript to a list of dictionaries
transcript_list = [
    {
        "text": snippet.text,
        "start": snippet.start,
        "duration": snippet.duration,
    }
    for snippet in transcript
]

print(transcript_list[:5])

[{'text': 'the following is a conversation with', 'start': 0.08, 'duration': 3.44}, {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96}, {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119}, {'text': 'a company that has published and builds', 'start': 6.72, 'duration': 4.48}, {'text': 'some of the most incredible artificial', 'start': 8.639, 'duration': 4.561}]


In [ ]:
# Display the full transcript list
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

In [ ]:
# Join transcript into a single string and split into chunks
transcript_text = " ".join(
    item["text"] for item in transcript_list
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript_text])

In [ ]:
# Display the number of chunks created
len(chunks)

168

In [ ]:
# Display the first chunk
chunks[0]

Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inter

In [ ]:
# Embedding Generation and Storing in vector store

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Display the mapping of indices to document IDs in the vector store
vector_store.index_to_docstore_id

{0: 'afb4c967-b609-43b0-ba66-ed1403c245e7',
 1: '3389dea6-fea3-4845-be13-84ad7a60eeaa',
 2: '17fb34a1-f1d3-4ca4-8ab4-6941979bc2b3',
 3: '14d7b90a-a89d-4c43-b8de-a188096f5f5b',
 4: '2a656619-6bd4-4bd5-bf40-b4a481840f92',
 5: '11ba3018-358a-491a-bba1-66259420525b',
 6: '91acadad-2469-4016-a094-cfb118c74428',
 7: '1c657e4e-cf55-4cc1-baab-5916ada2ef2c',
 8: 'a14543b3-8315-4a1e-890e-585ae9a9d3c0',
 9: '782c7bc5-6059-401f-83a2-14f5ef8f3d21',
 10: '14cf2182-e853-430a-a7e8-f4fc34ea71e5',
 11: '3c46631e-fb52-49e8-94ad-e96623d3fdd4',
 12: 'd33c8aea-837c-4b08-b37c-1164f43142ec',
 13: 'ac3fc7f7-b488-4fa5-a2a3-3428d3b5e4ae',
 14: '81ce76d9-4efc-4eb2-a6c0-3c98c8c5803e',
 15: '13347902-1b69-400d-87e9-1f387c270c1a',
 16: '95a6e9f0-1bb7-4236-8f2c-524aaadc8485',
 17: 'a9454235-f846-490c-8551-6d74e95c8cbb',
 18: 'e74a0815-6d0c-48af-9ff4-fd27f50f2876',
 19: 'bbd14a7b-71f4-4d06-bbb9-4821c1c0f74c',
 20: 'c1db7ce8-051f-4e8c-9002-075ba73d8573',
 21: 'cf384800-3b1f-45b9-8c48-2b7bc03b371c',
 22: '5f4afaa6-b623-

In [ ]:
# Attempt to retrieve documents by a specific ID (for demonstration)
vector_store.get_by_ids(['c6dc39a2-0f23-47a4-8e3d-10cbff59ce8a'])

[]

In [ ]:
# Create a retriever from the vector store
retriever = vector_store.as_retriever(search_type = 'similarity',search_kwargs = {"k":4})

In [ ]:
# Display the retriever object
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7af1d61ef8f0>, search_kwargs={'k': 4})

In [ ]:
# Invoke the retriever with a sample query
retriever.invoke('What is Deepmind?')

[Document(id='9e3f354b-8b4f-484b-9572-0766c6da0f03', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

In [ ]:
# Define a prompt template for the LLM
prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Use ONLY the information in the transcript below.

If the answer is not contained in the transcript, reply:
"I don't know."

Keep the answer:
- Clear
- Concise
- Accurate
- No markdown
- No bullet points unless requested

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

In [ ]:
# Retrieve documents relevant to the question about aliens
question = 'is the topic aliens discussed in this video? if yes then what was discussed'
retrieved_docs = retriever.invoke(question)

In [ ]:
# Display the retrieved documents for the alien question
retrieved_docs

[Document(id='a68a5ad6-c7ac-41af-9c9e-0f2e7a6d7ce4', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time'),
 Document(id='1c34813f-0c06-4343-b94b-5698d0180662', metadata={}, page_content="thoughts it could be some interactions with our mind that we think are originating from us is actually something that uh is coming from other life forms elsewhere consciousness itself might be that it could be but i don't see any sensible argument to the why why would all of the alien species be using this way yes some of them will be more primitive they would be close to our level you know there would there should be a whole sort of normal distribution of these things right some would be aggressive some would be you know curious others would be ve

In [ ]:
# Format retrieved documents into context text for the prompt
context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)

In [ ]:
# Display the formatted context text
context_text

"demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time\n\nthoughts it could be some interactions with our mind that we think are originating from us is actually something that uh is coming from other life forms elsewhere consciousness itself might be that it could be but i don't see any sensible argument to the why why would all of the alien species be using this way yes some of them will be more primitive they would be close to our level you know there would there should be a whole sort of normal distribution of these things right some would be aggressive some would be you know curious others would be very stoical and philosophical because you know maybe they're a million years older than us but it's not it shouldn't be like what i mean one one alien civilizat

In [ ]:
# Invoke the prompt with the context and question
final_prompt = prompt.invoke({'context':context_text, 'question':question})

In [ ]:
# Initialize the Hugging Face LLM endpoint
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-8B",
    temperature=0.2,
    max_new_tokens=512,
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [ ]:
# Invoke the LLM with the final prompt to get an answer
answer = llm.invoke(final_prompt)

In [ ]:
# Print the LLM's answer
print(answer.content)



Yes, the topic of aliens was discussed. The conversation explored the possibility that some interactions with human consciousness might originate from extraterrestrial life forms, suggesting consciousness could be a shared phenomenon. It also considered the diversity of alien civilizations, their potential communication methods, and the idea that advanced alien species might influence human thought or reality in ways we don't fully understand.


In [ ]:
# Building the Chain - Import LangChain Runnable components

from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Define a helper function to format retrieved documents
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
# Create a parallel chain to handle context and question
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [ ]:
# Invoke the parallel chain with a sample question
parallel_chain.invoke("who is demis?")

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [ ]:
# Initialize a string output parser
parser = StrOutputParser()

In [ ]:
# Build the main RAG chain
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
# Invoke the main chain to summarize the video
main_chain.invoke('Can you summarize the video ?')

'\n\nThe video features a discussion between Demis Hassabis and another individual, touching on topics like AI identity, work habits, and reflections on changes in research and productivity over time. It includes a quote from Edsger Dijkstra about computer science, questions about daily routines and work setups, and musings on fundamental explanations of physics beyond the standard model. The conversation also explores meta-Turing test concepts and the impact of AI on benchmarking.'